# SQL Practice - Day 13

## 15 SQL Interview Questions

**Focus:** CTE, Window Functions, JOINs, SELF JOIN, CASE WHEN

**Level:** 2–3 Year Data Analyst

Write your SQL in the blank cell under each question. No solutions are provided.

## Q1 — CTE

**Task:** Find each customer's total spending using a CTE.

In [1]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,103,104],'amount':[100,300,500,200,400,700]})
df

,customer_id,amount
0,101,100
1,101,300
2,102,500
3,103,200
4,103,400
5,104,700


In [3]:
import pandasql
from pandasql import sqldf


sqldf("""
with my_cte as (
select customer_id , sum(amount) as total_spending
from df
group by customer_id



)
select * from my_cte


""")

,customer_id,total_spending
0,101,400
1,102,500
2,103,600
3,104,700


## Q2 — CTE + CASE

**Task:** Using a CTE, calculate total spending per customer and classify customers as High (>=1000) or Low (<1000).

In [4]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104],'amount':[500,700,300,900,200,400]})
df

,customer_id,amount
0,101,500
1,101,700
2,102,300
3,103,900
4,104,200
5,104,400


In [10]:
sqldf("""
with my_cte as (
select customer_id , sum(amount) as total_spending
from df
group by customer_id

)
select customer_id , total_spending,
case 
    when total_spending >= 1000 then 'high'
    else 'low'
end as buket
from my_cte

""")

,customer_id,total_spending,buket
0,101,1200,high
1,102,300,low
2,103,900,low
3,104,600,low


## Q3 — CTE + Window

**Task:** Find the top 2 customers by total spending using a CTE and DENSE_RANK().

In [11]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,103,104,104,105],'amount':[500,500,1200,1500,700,800,1500]})
df

,customer_id,amount
0,101,500
1,101,500
2,102,1200
3,103,1500
4,104,700
5,104,800
6,105,1500


In [14]:
sqldf("""  
with my_cte as (
select customer_id , sum(amount) as total_spending,
        dense_rank()
        over(order by sum(amount) desc) as rn
        from df 
        group by customer_id 
)
select * from my_cte
where rn <= 2


""")

,customer_id,total_spending,rn
0,105,1500,1
1,104,1500,1
2,103,1500,1
3,102,1200,2


## Q4 — INNER JOIN

**Task:** Return employee name, department name and salary by joining Employees and Departments.

In [15]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4],'name':['A','B','C','D'],'dept_id':[10,10,20,30],'salary':[50000,60000,70000,80000]})
df1=pd.DataFrame({'dept_id':[10,20,30],'department':['HR','IT','Sales']})
print(df); print(df1)

   emp_id name  dept_id  salary
0       1    A       10   50000
1       2    B       10   60000
2       3    C       20   70000
3       4    D       30   80000
   dept_id department
0       10         HR
1       20         IT
2       30      Sales


In [20]:
sqldf("""
select e.name  as name , d.department as department , e.salary as salary
from df e
inner join df1 d on e.dept_id = d.dept_id




""")

,name,department,salary
0,A,HR,50000
1,B,HR,60000
2,C,IT,70000
3,D,Sales,80000


## Q5 — LEFT JOIN

**Task:** Find all customers and their total order amount. Customers with no orders must also appear.

In [21]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103,104],'name':['A','B','C','D']})
df1=pd.DataFrame({'order_id':[1,2,3,4],'customer_id':[101,101,103,103],'amount':[100,200,300,400]})
print(df); print(df1)

   customer_id name
0          101    A
1          102    B
2          103    C
3          104    D
   order_id  customer_id  amount
0         1          101     100
1         2          101     200
2         3          103     300
3         4          103     400


In [27]:
sqldf("""
select c.name as name , o.amount as total_order_amout
from df c
left join df1 o on c.customer_id = o.customer_id 
group by name


""")

,name,total_order_amout
0,A,100.0
1,B,NaN
2,C,300.0
3,D,NaN


## Q6 — SELF JOIN

**Task:** Find employees whose salary is greater than their manager's salary.

In [28]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['ManagerA','EmpB','EmpC','ManagerD','EmpE'],'manager_id':[None,1,1,None,4],'salary':[80000,90000,70000,70000,75000]})
df

,emp_id,name,manager_id,salary
0,1,ManagerA,NaN,80000
1,2,EmpB,1.0,90000
2,3,EmpC,1.0,70000
3,4,ManagerD,NaN,70000
4,5,EmpE,4.0,75000


In [32]:
sqldf("""
select e.name as emp_name, e.salary  as emp_salary , m.name as manager_name , m.salary as manager_salary
from df e
join df m on e.manager_id = m.manager_id 
where e.salary > m.salary



""")

,emp_name,emp_salary,manager_name,manager_salary
0,EmpB,90000,EmpC,70000


## Q7 — SELF JOIN

**Task:** Return employee name and manager name for every employee who has a manager.

In [33]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5],'name':['A','B','C','D','E'],'manager_id':[None,1,1,4,4]})
df

,emp_id,name,manager_id
0,1,A,NaN
1,2,B,1.0
2,3,C,1.0
3,4,D,4.0
4,5,E,4.0


In [42]:
sqldf("""
select e.name as emp_name , m.name as manager_name
from df e
 left join df m on e.manager_id = e.manager_id
 where e.emp_id < m.emp_id

""")

,emp_name,manager_name
0,B,C
1,B,D
2,B,E
3,C,D
4,C,E
5,D,E


## Q8 — Window

**Task:** Find the highest-paid employee in each department. Return all ties.

In [43]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F'],'department':['HR','HR','IT','IT','Sales','Sales'],'salary':[60000,80000,90000,90000,90000,85000]})
df

,name,department,salary
0,A,HR,60000
1,B,HR,80000
2,C,IT,90000
3,D,IT,90000
4,E,Sales,90000
5,F,Sales,85000


In [47]:
sqldf("""
select * 
from (select * , 
        dense_rank()
        over(partition by department order by salary desc) as rn
        from df) t
where rn = 1



""")

,name,department,salary,rn
0,B,HR,80000,1
1,C,IT,90000,1
2,D,IT,90000,1
3,E,Sales,90000,1


## Q9 — Window

**Task:** For each customer, show the current order amount and previous order amount using LAG().

In [48]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,101,102,102],'order_date':['2025-01-01','2025-02-01','2025-03-01','2025-01-10','2025-02-10'],'amount':[100,300,200,500,700]})
df

,customer_id,order_date,amount
0,101,2025-01-01,100
1,101,2025-02-01,300
2,101,2025-03-01,200
3,102,2025-01-10,500
4,102,2025-02-10,700


## Q10 — Window + CASE

**Task:** For each employee, calculate department average salary using a window function and classify them as Above Average or Below Average.

In [50]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E'],'department':['HR','HR','IT','IT','IT'],'salary':[50000,70000,60000,90000,65000]})
df

,name,department,salary
0,A,HR,50000
1,B,HR,70000
2,C,IT,60000
3,D,IT,90000
4,E,IT,65000


In [51]:
sqldf("""
with my_cte as (
select name , departmnet , salary , avg(b.salary) as avg_dept_salary



)



""")

## Q11 — CASE

**Task:** Classify orders as High (>=1000), Medium (500–999), or Low (<500).

In [52]:
import pandas as pd
df=pd.DataFrame({'order_id':[1,2,3,4,5],'amount':[300,500,800,1000,1500]})
df

,order_id,amount
0,1,300
1,2,500
2,3,800
3,4,1000
4,5,1500


In [54]:
sqldf("""
select order_id , amount,
case
    when amount >= 1000 then 'High'
    when amount >=500 and amount <999 then 'Medium'
    else 'Low'
end as bulet
from df


""")

,order_id,amount,bulet
0,1,300,Low
1,2,500,Medium
2,3,800,Medium
3,4,1000,High
4,5,1500,High


## Q12 — CASE + Aggregation

**Task:** For each customer, count Premium orders (>=1000) and Regular orders (<1000).

In [55]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,101,102,102,103,103],'amount':[500,1500,1200,300,800,2000]})
df

,customer_id,amount
0,101,500
1,101,1500
2,102,1200
3,102,300
4,103,800
5,103,2000


In [59]:
sqldf("""
select customer_id , sum(amount) as amount,
case 
    when amount >= 1000 then 'premimun'
    else 'regular'
end as buket
from df
group by customer_id


""")

,customer_id,amount,buket
0,101,2000,regular
1,102,1500,premimun
2,103,2800,regular


## Q13 — JOIN + CASE

**Task:** Join customers and orders, then classify each customer's total spending as Gold (>=2000), Silver (1000–1999), or Bronze (<1000).

In [60]:
import pandas as pd
df=pd.DataFrame({'customer_id':[101,102,103],'name':['A','B','C']})
df1=pd.DataFrame({'order_id':[1,2,3,4,5],'customer_id':[101,101,102,103,103],'amount':[500,1800,1200,300,400]})
print(df); print(df1)

   customer_id name
0          101    A
1          102    B
2          103    C
   order_id  customer_id  amount
0         1          101     500
1         2          101    1800
2         3          102    1200
3         4          103     300
4         5          103     400


In [71]:
sqldf("""
select c.name as name , o.order_id as order_id , sum(o.amount) as total_amount,
case
    when sum(o.amount) >= 2000 then 'Gold'
    when sum(o.amount) >=1000 and sum(o.amount) <= 1999 then 'sliver'
    else 'bronze'
end as buket

from df c
inner join df1 o on c.customer_id = o.customer_id
group by c.customer_id 


""")

,name,order_id,total_amount,buket
0,A,1,2300,Gold
1,B,3,1200,sliver
2,C,4,700,bronze


## Q14 — CTE + SELF JOIN

**Task:** Using a CTE to calculate department average salary, find employees who earn more than their department average and whose salary is also greater than their manager's salary.

In [72]:
import pandas as pd
df=pd.DataFrame({'emp_id':[1,2,3,4,5,6],'name':['A','B','C','D','E','F'],'department':['HR','HR','IT','IT','Sales','Sales'],'manager_id':[None,1,None,3,None,5],'salary':[80000,90000,70000,90000,80000,90000]})
df

,emp_id,name,department,manager_id,salary
0,1,A,HR,NaN,80000
1,2,B,HR,1.0,90000
2,3,C,IT,NaN,70000
3,4,D,IT,3.0,90000
4,5,E,Sales,NaN,80000
5,6,F,Sales,5.0,90000


In [73]:
sqldf("""
with my_cte as (





)


""")

## Q15 — Mixed Interview

**Task:** Find the top 2 highest-paid employees in each department, show their department average salary, and classify each selected employee as Above Average or At Average using CASE.

In [ ]:
import pandas as pd
df=pd.DataFrame({'name':['A','B','C','D','E','F','G'],'department':['HR','HR','HR','IT','IT','Sales','Sales'],'salary':[50000,70000,70000,60000,90000,80000,90000]})
df

In [ ]:
# Write your SQL here